# Feature Engineering

Material ONIA — Resurse educaționale. Exercițiile pot fi modificate și executate în JupyterLab.

## Ce este o caracteristică (feature)?

O caracteristică este o informație folosită de model pentru a face o predicție: temperatura, lungimea unui text, vârsta unui obiect, culoarea unui produs. Dacă avem 100 de fotografii și calculăm pentru fiecare trei valori numerice, matricea de intrare X are 100 de rânduri și 3 coloane. Feature engineering înseamnă crearea, selectarea, transformarea sau combinarea caracteristicilor astfel încât problema să fie mai simplu de învățat. Alegem transformările pe baza semnificației datelor și le evaluăm corect, nu doar pentru că o transformare aduce un scor public puțin mai bun.

In [ ]:
X = [
    [20, 2, 300],
    [25, 3, 450],
    [30, 1, 500],
]
print('Observații:', len(X))
print('Caracteristici:', len(X[0]))

## O reprezentare nepotrivită poate bloca un model

Să presupunem două caracteristici x₁ și x₂ și o clasă determinată de semnul produsului x₁·x₂. În plan, cazurile pozitive apar în două cadrane opuse, deci nu pot fi separate de o singură dreaptă. Dacă adăugăm caracteristica z=x₁*x₂ , regula devine simplă: z>0 înseamnă clasa pozitivă. Aceasta este o demonstrație de feature engineering, nu o garanție că multiplicarea variabilelor ajută în orice problemă.

In [ ]:
date = [(-2, -1), (-1, 2), (2, 2), (1, -2)]
for x1, x2 in date:
    z = x1 * x2
    clasa = int(z > 0)
    print((x1, x2), 'z=', z, 'clasă=', clasa)

## Scalare: standardizare și Min–Max

Unele modele folosesc direct distanțele sau optimizarea numerică: K-NN, SVM, regresie logistică regularizată, rețele neurale. Dacă o caracteristică variază între 0 și 1, iar alta între 0 și 100 000, cea din urmă poate domina distanța Euclidiană. StandardScaler transformă fiecare caracteristică numerică în (x − media_train)/abaterea_train ; MinMaxScaler o transformă prin raportare la minimul și maximul observat în train. Păstrează setul de validare/test în afara lui fit . Valorile transformate pot ieși în afara intervalului [0,1] pe test: nu este automat o eroare.

In [ ]:
from sklearn.preprocessing import StandardScaler
X_train = [[1, 1000], [2, 1500], [3, 2000]]
X_test = [[4, 2500]]
scaler = StandardScaler()
print(scaler.fit_transform(X_train).round(2))
print(scaler.transform(X_test).round(2))

## Transformări polinomiale

Într-o regresie, un model liniar în coeficienți poate învăța relații neliniare în intrări prin caracteristici noi: x², x³, x₁x₂. De exemplu, y ≈ w₀ + w₁x + w₂x² este o parabolă în geometrie, dar este liniar față de coeficienții w. PolynomialFeatures construiește automat aceste coloane. Numărul caracteristicilor poate crește rapid cu gradul și numărul variabilelor, ceea ce mărește riscul de overfitting; de aceea transformarea se combină adesea cu regularizarea.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
X = [[2, 3], [4, 5]]
poly = PolynomialFeatures(degree=2, include_bias=False)
print(poly.fit_transform(X))
print(poly.get_feature_names_out(['x1', 'x2']))

## Categorii nominale: one-hot encoding

Dacă variabila „culoare” poate fi roșu, verde sau albastru, codificarea 0, 1, 2 introduce o ordine și o distanță care nu există în semnificația categoriei. One-hot encoding construiește câte o coloană indicator pentru fiecare categorie: o singură poziție este 1, celelalte 0. Pentru date noi trebuie prevăzut și cazul unei categorii necunoscute, de exemplu handle_unknown="ignore" . One-hot crește numărul coloanelor. Pentru cardinalitate foarte mare putem avea nevoie de alte reprezentări sau de modele care gestionează categorii nativ.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
X_train = [['roșu'], ['albastru'], ['verde'], ['roșu']]
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
print(enc.fit_transform(X_train))
print(enc.transform([['galben']]))

## Categorii ordinale și valori numerice

O variabilă precum „mic / mediu / mare” are o ordine naturală. O codificare ordinală poate avea sens, dar distanțele numerice dintre niveluri nu sunt neapărat egale. Dacă folosim 0,1,2, modelul ar putea interpreta diferența dintre mic și mediu ca egală cu cea dintre mediu și mare. Distinge etichetele unei clase țintă y de caracteristicile categorice din X . Nu folosi LabelEncoder ca soluție universală pentru toate coloanele din X.

In [ ]:
ordine = {'mic': 0, 'mediu': 1, 'mare': 2}
marimi = ['mare', 'mic', 'mediu']
coduri = [ordine[m] for m in marimi]
print(coduri)

## Caracteristici extrase din șiruri și date calendaristice

Dintr-un text putem extrage lungimea, numărul de cuvinte, frecvențe de termeni sau reprezentări TF-IDF. Dintr-o dată calendaristică putem extrage ziua săptămânii și luna. Pentru fenomene ciclice, cum sunt orele dintr-o zi, transformarea sin/cos poate reprezenta corect apropierea dintre ora 23 și ora 0. Atenție la datele pe care nu le-am fi cunoscut în momentul predicției: un câmp creat după apariția evenimentului de prezis poate produce data leakage.

In [ ]:
import math
ore = [0, 6, 12, 18, 23]
for h in ore:
    unghi = 2 * math.pi * h / 24
    print(h, round(math.sin(unghi), 3), round(math.cos(unghi), 3))

## Combinarea coloanelor eterogene

Un set de date real poate conține simultan vârstă, venit, oraș și abonament. Folosim ColumnTransformer pentru transformări diferite pe grupuri de coloane: imputare și scalare pe numerice, imputare și one-hot pe categorii. Apoi trecem rezultatele într-un model prin Pipeline . Avantaj: procesul este reproductibil și evită recalcularea independentă a transformărilor pe datele noi.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
X = pd.DataFrame({'vârstă': [15, 16, 17], 'oraș': ['Iași', 'Cluj', 'Iași']})
pre = ColumnTransformer([
    ('numeric', make_pipeline(SimpleImputer(strategy='median'),
                              StandardScaler()), ['vârstă']),
    ('categoric', OneHotEncoder(handle_unknown='ignore'), ['oraș'])
])
print(pre.fit_transform(X))

## Selecția caracteristicilor

Mai multe caracteristici nu înseamnă automat un model mai bun. Coloanele neinformative pot crește costul și riscul de overfitting. Selectarea caracteristicilor poate fi făcută prin cunoștințe de domeniu, analize univariate, regularizare L1 sau metode de tip SelectKBest . Regulă esențială: dacă selectarea caracteristicilor folosește etichetele țintă, selectarea trebuie să aibă loc în interiorul fiecărui fold de antrenare , nu înainte de cross-validation.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectKBest, f_classif
X, y = load_iris(return_X_y=True)
select = SelectKBest(score_func=f_classif, k=2)
X_redus = select.fit_transform(X, y)
print(X.shape, X_redus.shape)

## Data leakage prin target encoding

Dacă pentru fiecare oraș înlocuim numele orașului cu media rezultatelor folosind și observațiile de validare , modelul poate „vedea” indirect răspunsurile pe care ar trebui să le prezică. Tehnicile de target encoding trebuie aplicate cu multă atenție, folosind estimări obținute doar din datele de antrenare, uneori cu schemă cross-fitting. În olimpiadă, un scor public neobișnuit de mare poate semnala o problemă de validare; investighează înainte de a presupune că ai descoperit un algoritm excepțional.

## Când merită să creezi noi caracteristici?

Introdu caracteristici noi când există un motiv: geometria problemei, semnificația fizică, un raport între cantități, o frecvență de evenimente, o agregare pe istoricul disponibil sau o reprezentare text. Compară modelul de bază cu modelul extins folosind aceleași folduri . Dacă diferența este mică și instabilă, merită preferată o soluție simplă și robustă. În laborator poți vedea cum o interacțiune de forma x₁·x₂ face separabil un set de clase care nu poate fi separat liniar în planul inițial.